# 02 - Leakage audit and train/serve replay

This audit verifies that features use only strictly earlier events and that fraud cases do not span train, validation, and test boundaries.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data" / "mock_full"
from momo_fraud.io import load_tables, load_yaml_config
tables = load_tables(DATA_DIR, PROJECT_ROOT / "configs" / "columns.yaml")
transactions = tables["transactions"]
assert len(transactions) == 100023
print("Audit dataset:", len(transactions), "rows")

Audit dataset: 100023 rows


## Replay features in timestamp order

Labels, fraud roles, scenarios, and case IDs are never passed to the feature builder.

In [2]:
from momo_fraud.features.state import NUMBER_COLUMNS, build_features
feature_config = load_yaml_config(PROJECT_ROOT / "configs" / "features.yaml")
features = build_features(tables, config=feature_config)
forbidden = {"is_fraud", "fraud_scenario", "fraud_role", "fraud_case"}
assert forbidden.isdisjoint(features.columns)
assert set(NUMBER_COLUMNS).issubset(features.columns)
print(f"Generated {len(features):,} point-in-time rows")
features[["tx_id", "timestamp", "payer_tx_count_1h", "payer_new_payee", "call_before_tx", "payer_credit_claim_mismatch_1h"]].head(10)

Generated 100,023 point-in-time rows


,tx_id,timestamp,payer_tx_count_1h,payer_new_payee,call_before_tx,payer_credit_claim_mismatch_1h
tx_id,,,,,,
TX0000000,TX0000000,2026-08-22 00:02:26+00:00,0,1,0,0
TX0000001,TX0000001,2026-08-22 00:02:40+00:00,0,1,0,0
TX0000002,TX0000002,2026-08-22 00:08:06+00:00,0,1,0,0
TX0000003,TX0000003,2026-08-22 00:10:43+00:00,0,1,0,0
TX0000004,TX0000004,2026-08-22 00:12:12+00:00,0,1,0,0
TX0000005,TX0000005,2026-08-22 00:12:40+00:00,0,1,0,0
TX0000006,TX0000006,2026-08-22 00:13:33+00:00,1,1,0,0
TX0000007,TX0000007,2026-08-22 00:16:08.806480+00:00,0,1,0,0
TX0000008,TX0000008,2026-08-22 00:17:14+00:00,0,1,0,0


## Check chronological and fraud-case splits

A case crossing a split boundary is excluded rather than split across train and test.

In [3]:
from momo_fraud.splits import time_case_split
assignment, boundaries = time_case_split(transactions)
print("Boundaries:", boundaries)
display(assignment.value_counts().rename_axis("split").to_frame("rows"))
case_splits = pd.DataFrame({"fraud_case": transactions["fraud_case"], "split": assignment}).dropna(subset=["fraud_case"])
assert case_splits.groupby("fraud_case")["split"].nunique().max() == 1

Boundaries: {'train_end': Timestamp('2026-10-14 10:23:14+0000', tz='UTC'), 'validation_end': Timestamp('2026-11-01 05:56:50.600000+0000', tz='UTC')}


,rows
split,
train,59974
test,19979
validation,19947
excluded,123


## Run replay invariant tests

In [4]:
import subprocess
subprocess.run([sys.executable, "-m", "pytest", "-q",
                str(PROJECT_ROOT / "tests" / "test_features_point_in_time.py"),
                str(PROJECT_ROOT / "tests" / "test_train_serve_parity.py"),
                str(PROJECT_ROOT / "tests" / "test_splits.py")],
               check=True, cwd=PROJECT_ROOT)

CompletedProcess(args=['C:\\Users\\yasmine\\AppData\\Local\\Programs\\Python\\Python314\\python.exe', '-m', 'pytest', '-q', 'C:\\Users\\yasmine\\Desktop\\Momo fraud hackathon\\FDE\\tests\\test_features_point_in_time.py', 'C:\\Users\\yasmine\\Desktop\\Momo fraud hackathon\\FDE\\tests\\test_train_serve_parity.py', 'C:\\Users\\yasmine\\Desktop\\Momo fraud hackathon\\FDE\\tests\\test_splits.py'], returncode=0)